# 02b · Árboles de decisión: la geometría de las particiones

**Módulo 4 · Sesión 10** — Árboles de decisión y bagging

## Objetivos

El notebook 03 construye un árbol a mano y lo lleva hasta bagging y Random Forest. Este
notebook va antes y mira el árbol **como geometría**, con los mismos 20 estudiantes de
los notebooks 01b y 01c:

1. Una pregunta "¿$x_j \leq t$?" es un **corte** del plano, y elegir $t$ es recorrer una
   curva de impureza **escalonada** buscando su punto más bajo.
2. El árbol completo es una partición del plano en **rectángulos**, y su predicción, una
   función en **terrazas**: compárese con la rampa de la logística.
3. Por qué los árboles **no necesitan escalar**: solo les importa el orden de los valores.
   Justo lo contrario de KNN y la SVM.
4. Lo que un corte paralelo a los ejes no puede hacer: una frontera **diagonal** necesita
   una escalera de hojas; la misma frontera, rotada, un solo corte.
5. La **miopía** del algoritmo voraz: en un XOR ningún primer corte parece útil, aunque
   dos cortes lo resuelven.
6. 🔵 Opcional: el árbol de regresión con los 10 estudiantes del módulo 3. Sus hojas
   predicen la media, y el corte se elige minimizando el mismo **área de cuadrados**.

**Paquetes:** `numpy`, `pandas`, `matplotlib`, `scikit-learn`.

In [ ]:
# Arranque para Google Colab (en local no hace nada): trae el repositorio para que
# ../datos y ../src existan. Ejecútala antes que cualquier otra celda.
import sys
if "google.colab" in sys.modules:
    !git clone -q --depth 1 https://github.com/delany-ramirez/machine_learning /content/machine_learning
    %cd /content/machine_learning/modulo-4-clasificacion-ensambles/notebooks

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import ListedColormap
from matplotlib.patches import Rectangle
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree

SEMILLA = 42
rng = np.random.default_rng(SEMILLA)

## 1. Los 20 estudiantes, sin estandarizar

En los notebooks 01b y 01c las variables tenían que estandarizarse. Aquí **no**: usamos el
promedio y las horas de estudio en sus unidades originales. La sección 5 explica por qué
un árbol puede darse ese lujo.

In [ ]:
datos = pd.read_csv("../datos/rendimiento-estudiantes.csv")
VARIABLES = ["promedio_anterior", "horas_estudio_semana"]

muestra = datos.sample(n=20, random_state=SEMILLA)
X = muestra[VARIABLES].to_numpy()
y = muestra["aprobo"].to_numpy()

COLORES = {0: "tab:red", 1: "tab:blue"}
FONDO = ListedColormap(["#f4c7c3", "#c6dbef"])
LIM_X, LIM_Y = (2.6, 5.0), (0.0, 17.0)


def preparar_eje(eje, titulo=""):
    eje.set_xlim(*LIM_X)
    eje.set_ylim(*LIM_Y)
    eje.set_xlabel("promedio_anterior")
    eje.set_ylabel("horas_estudio_semana")
    eje.set_title(titulo)


def dibujar_puntos(eje, X, y, s=45):
    for clase, nombre in [(0, "no aprobó"), (1, "aprobó")]:
        eje.scatter(*X[y == clase].T, color=COLORES[clase], s=s, edgecolor="black",
                    linewidth=0.5, label=nombre, zorder=3)


fig, eje = plt.subplots(figsize=(6, 5))
dibujar_puntos(eje, X, y)
eje.legend(loc="upper right")
preparar_eje(eje, "20 estudiantes, en unidades originales")
plt.show()

## 2. Una pregunta es un corte

La primera pregunta del árbol tiene la forma "¿$x_j \leq t$?". En el plano es una recta
**vertical** (si pregunta por el promedio) u **horizontal** (si pregunta por las horas)
que parte a los 20 estudiantes en dos grupos. Para decidir cuál, el árbol mide la
impureza de Gini de cada lado, $G = 1 - p_0^2 - p_1^2$, y la promedia ponderando por el
tamaño de cada grupo:

$$
G_{\text{corte}}(j, t) = \frac{n_I}{n}\,G(\text{izquierda}) + \frac{n_D}{n}\,G(\text{derecha}).
$$

El mejor corte es el que deja esa impureza ponderada más baja. Recorramos todos los
umbrales posibles de cada variable.

In [ ]:
def gini(y):
    if len(y) == 0:
        return 0.0
    p = np.mean(y)
    return 1 - p**2 - (1 - p) ** 2


def gini_corte(x, y, t):
    izq, der = y[x <= t], y[x > t]
    return (len(izq) * gini(izq) + len(der) * gini(der)) / len(y)


def candidatos(x):
    """Puntos medios entre valores consecutivos distintos: los únicos umbrales que importan."""
    v = np.unique(x)
    return (v[:-1] + v[1:]) / 2


mejores = {}
fig, ejes = plt.subplots(2, 2, figsize=(13, 8), gridspec_kw={"height_ratios": [3, 2]})
for col, (j, nombre) in enumerate(zip([0, 1], VARIABLES)):
    lim = LIM_X if j == 0 else LIM_Y
    ts = np.linspace(*lim, 1000)
    curva = [gini_corte(X[:, j], y, t) for t in ts]
    cands = candidatos(X[:, j])
    valores = np.array([gini_corte(X[:, j], y, t) for t in cands])
    t_mejor = cands[np.argmin(valores)]
    mejores[nombre] = (valores.min(), t_mejor)

    eje = ejes[0, col]
    dibujar_puntos(eje, X, y)
    if j == 0:
        eje.axvline(t_mejor, color="black", linewidth=2)
    else:
        eje.axhline(t_mejor, color="black", linewidth=2)
    preparar_eje(eje, f"Mejor corte en {nombre}: t = {t_mejor:.2f}")

    eje = ejes[1, col]
    eje.plot(ts, curva, color="gray")
    eje.plot(cands, valores, "o", color="black", markersize=4, label="puntos medios")
    eje.plot(t_mejor, valores.min(), "o", color="gold", markersize=10, markeredgecolor="black",
             label=f"mínimo: {valores.min():.3f}")
    eje.axhline(gini(y), color="gray", linestyle="--", linewidth=1, label=f"sin cortar: {gini(y):.3f}")
    eje.set_xlim(*lim)
    eje.set_xlabel(f"umbral t sobre {nombre}")
    eje.set_ylabel("Gini ponderado")
    eje.legend(fontsize=9)
plt.tight_layout()
plt.show()

for nombre, (g, t) in mejores.items():
    print(f"{nombre:>22}: mejor t = {t:.3f}, Gini ponderado = {g:.4f}")

tocon = DecisionTreeClassifier(max_depth=1, random_state=SEMILLA).fit(X, y)
print(f"\nscikit-learn elige: {VARIABLES[tocon.tree_.feature[0]]} <= {tocon.tree_.threshold[0]:.3f}")

Dos cosas se leen en las curvas de abajo:

- Son **escalonadas**. Mientras el umbral se mueve entre dos estudiantes consecutivos,
  nadie cambia de lado y la impureza no cambia. Por eso basta evaluar un candidato por
  escalón, el punto medio, y la búsqueda es exhaustiva y barata. Es un contraste con el
  paraboloide suave del módulo 3: aquí no hay gradiente que seguir, solo escalones que
  recorrer.
- Los dos mínimos están cerca, pero el de las horas (0.175) es más bajo que el del
  promedio (0.200): la primera pregunta será "¿horas ≤ 3.2?", que aparta a cuatro no
  aprobados que estudian muy poco, y coincide con la de `scikit-learn`. El promedio
  entra en el segundo nivel (sección 3). El árbol no "sabe" que el promedio es la
  variable más predictiva en general: solo ve qué corte limpia más **estos** 20 puntos.
  Con otros 20 estudiantes la raíz podría ser otra; es la inestabilidad que mide el
  notebook 03.

## 3. El árbol crece: rectángulos dentro de rectángulos

Tras el primer corte, cada lado se trata como un problema nuevo y se corta otra vez, con
la misma búsqueda, pero **solo dentro de su rectángulo**. Cada pregunta subdivide una
región; ninguna atraviesa todo el plano. Leamos las hojas directamente de la estructura
interna de `DecisionTreeClassifier` y dibujemos sus rectángulos.

In [ ]:
def rectangulos_hojas(arbol, limites):
    """Lista de (x0, x1, y0, y1, n_clase0, n_clase1) para cada hoja del árbol."""
    t = arbol.tree_
    hojas = []

    def recorrer(nodo, lim):
        if t.children_left[nodo] == -1:  # es hoja
            conteo = t.value[nodo][0] / t.value[nodo][0].sum() * t.n_node_samples[nodo]
            hojas.append((*lim, *np.round(conteo).astype(int)))
            return
        j, umbral = t.feature[nodo], t.threshold[nodo]
        izq, der = list(lim), list(lim)
        izq[2 * j + 1] = umbral  # el lado izquierdo es x_j <= umbral
        der[2 * j] = umbral
        recorrer(t.children_left[nodo], izq)
        recorrer(t.children_right[nodo], der)

    recorrer(0, list(limites))
    return hojas


def dibujar_particion(eje, arbol, X, y, titulo):
    for x0, x1, y0, y1, n0, n1 in rectangulos_hojas(arbol, (*LIM_X, *LIM_Y)):
        clase = int(n1 > n0)
        pureza = max(n0, n1) / (n0 + n1)
        eje.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, facecolor=COLORES[clase],
                                alpha=0.15 + 0.35 * (pureza - 0.5) / 0.5, edgecolor="black",
                                linewidth=1.2, zorder=1))
        eje.text((x0 + x1) / 2, (y0 + y1) / 2, f"{n0}|{n1}", ha="center", va="center",
                 fontsize=9, zorder=4, bbox=dict(facecolor="white", alpha=0.7, linewidth=0))
    dibujar_puntos(eje, X, y, s=30)
    preparar_eje(eje, titulo)


fig, ejes = plt.subplots(1, 4, figsize=(20, 4.8))
for eje, d in zip(ejes, [1, 2, 3, None]):
    a = DecisionTreeClassifier(max_depth=d, random_state=SEMILLA).fit(X, y)
    dibujar_particion(eje, a, X, y, f"profundidad {d or 'sin límite'}: {a.get_n_leaves()} hojas")
plt.tight_layout()
plt.show()

Cada rectángulo es una hoja, con el conteo "no aprobó | aprobó" de los estudiantes que
caen en ella; el color más intenso indica una hoja más pura. Sin límite de profundidad el
árbol sigue cortando hasta que **todas** las hojas son puras, y algunos rectángulos
terminan conteniendo a un solo estudiante: es el sobreajuste que el notebook 03 mide con
la profundidad.

El diagrama de árbol y la partición son dos dibujos de lo mismo. Para la profundidad 2:

In [ ]:
arbol2 = DecisionTreeClassifier(max_depth=2, random_state=SEMILLA).fit(X, y)
fig, ejes = plt.subplots(1, 2, figsize=(15, 5), gridspec_kw={"width_ratios": [1.4, 1]})
plot_tree(arbol2, feature_names=VARIABLES, class_names=["no aprobó", "aprobó"], filled=True,
          impurity=True, ax=ejes[0], fontsize=9)
dibujar_particion(ejes[1], arbol2, X, y, "La misma partición en el plano")
plt.tight_layout()
plt.show()

Cada camino de la raíz a una hoja es una conjunción de desigualdades ("promedio > t₁ **y**
horas ≤ t₂"), es decir, un rectángulo. Leer un árbol pequeño es leer una lista de reglas,
y es lo que lo vuelve interpretable.

## 4. La predicción es una función en terrazas

En las hojas, el árbol predice la **fracción** de aprobados: una probabilidad constante
en todo el rectángulo. Como superficie sobre el plano, la probabilidad de un árbol es un
conjunto de **terrazas** planas con escalones verticales entre ellas. La logística, en
cambio, es una **rampa** suave: la sigmoide de una combinación lineal. Con los 400
estudiantes:

In [ ]:
X_todos = datos[VARIABLES].to_numpy()
y_todos = datos["aprobo"].to_numpy()

arbol_400 = DecisionTreeClassifier(max_depth=3, random_state=SEMILLA).fit(X_todos, y_todos)
logistica_400 = make_pipeline(StandardScaler(), LogisticRegression()).fit(X_todos, y_todos)

g1, g2 = np.meshgrid(np.linspace(2.0, 5.0, 150), np.linspace(0, 30, 150))
malla = np.c_[g1.ravel(), g2.ravel()]

fig = plt.figure(figsize=(15, 6))
for k, (modelo, titulo) in enumerate([(arbol_400, "Árbol (profundidad 3): terrazas"),
                                      (logistica_400, "Logística: una rampa")]):
    p = modelo.predict_proba(malla)[:, 1].reshape(g1.shape)
    eje3d = fig.add_subplot(1, 2, k + 1, projection="3d")
    eje3d.plot_surface(g1, g2, p, cmap="RdBu", vmin=0, vmax=1, rstride=2, cstride=2,
                       linewidth=0, alpha=0.9)
    eje3d.set_xlabel("promedio_anterior")
    eje3d.set_ylabel("horas_estudio")
    eje3d.set_zlabel("P(aprueba)")
    eje3d.set_zlim(0, 1)
    eje3d.view_init(elev=25, azim=-130)
    eje3d.set_title(titulo)
plt.tight_layout()
plt.show()

El árbol de profundidad 3 tiene a lo sumo 8 hojas, así que su probabilidad toma a lo sumo
8 valores — como KNN, con su escalera de $k+1$ peldaños (notebook 01b), pero con
escalones rectos y paralelos a los ejes. Dos consecuencias prácticas:

- **Un árbol solo no extrapola ni interpola**: más allá del último corte la terraza sigue
  plana para siempre, y entre dos cortes no hay transición.
- **Promediar árboles suaviza las terrazas.** Cientos de árboles con cortes en lugares
  ligeramente distintos producen cientos de escalones pequeños que, sumados, se parecen a
  una rampa. Es lo que hacen bagging y Random Forest (notebook 03) y, de otra forma,
  boosting (notebook 05).

## 5. Los árboles no necesitan escalar

Un corte "¿$x_j \leq t$?" solo depende de **qué estudiantes quedan a cada lado**, es
decir, del **orden** de los valores de $x_j$. Cualquier transformación que respete ese
orden —estandarizar, tomar logaritmos, elevar al cubo— deja exactamente los mismos
candidatos (en otras unidades) y los mismos grupos. KNN, en cambio, mide distancias, y
cualquier cambio de escala cambia quién es vecino de quién (notebook 01b, sección 6).

Comprobémoslo con los 400 estudiantes y tres versiones de las mismas dos variables, con
las mismas particiones de validación cruzada para todos:

In [ ]:
versiones = {
    "originales": X_todos,
    "estandarizadas": (X_todos - X_todos.mean(axis=0)) / X_todos.std(axis=0),
    "promedio³ y log(1 + horas)": np.c_[X_todos[:, 0] ** 3, np.log1p(X_todos[:, 1])],
}
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEMILLA)

ref_arbol = DecisionTreeClassifier(random_state=SEMILLA).fit(X_todos, y_todos)
ref_cv = {
    "árbol": cross_val_predict(DecisionTreeClassifier(random_state=SEMILLA), X_todos, y_todos, cv=cv),
    "KNN (k=15)": cross_val_predict(KNeighborsClassifier(15), X_todos, y_todos, cv=cv),
}
filas = []
for nombre, Xv in versiones.items():
    arbol_v = DecisionTreeClassifier(random_state=SEMILLA).fit(Xv, y_todos)
    fila = {
        "variables": nombre,
        "árbol: hojas": arbol_v.get_n_leaves(),
        "árbol: mismas predicciones en entrenamiento": np.mean(arbol_v.predict(Xv) == ref_arbol.predict(X_todos)),
    }
    for modelo, ref in ref_cv.items():
        base = DecisionTreeClassifier(random_state=SEMILLA) if modelo == "árbol" else KNeighborsClassifier(15)
        pred = cross_val_predict(base, Xv, y_todos, cv=cv)
        fila[f"{modelo}: mismas predicciones en validación"] = np.mean(pred == ref)
    filas.append(fila)
pd.DataFrame(filas).set_index("variables").round(4)

En entrenamiento el árbol es **idéntico** en las tres versiones: mismas hojas, mismas
predicciones. En validación coincide en el 99 % de los casos; la diferencia minúscula
viene de que el punto medio entre dos valores no se transforma igual que los valores (el
punto medio de dos logaritmos no es el logaritmo del punto medio), y un estudiante de
validación que cae justo en ese hueco puede quedar del otro lado. KNN, en cambio, cambia
más de una predicción de cada diez solo por estandarizar.

Esto vale para cualquier modelo construido con árboles —Random Forest, XGBoost,
LightGBM—: **escalar no les hace nada**, y los valores atípicos en $x$ tampoco los
arrastran, porque un valor extremo solo cuenta por su posición en el orden. Por eso
ningún modelo de árboles del notebook 04 lleva `StandardScaler` (la logística con la que
se comparan, sí).

## 6. Solo cortes paralelos a los ejes

El precio de preguntar por una variable a la vez es que cada frontera está hecha de
segmentos **verticales y horizontales**. Si la frontera verdadera es una diagonal —por
ejemplo, "aprueba si $x_2 > x_1$"— el árbol solo puede aproximarla con una **escalera**.
Un ejemplo sintético, sin ruido, con 400 puntos en el cuadrado unitario:

In [ ]:
X_diag = rng.uniform(size=(400, 2))
y_diag = (X_diag[:, 1] > X_diag[:, 0]).astype(int)
X_diag_prueba = rng.uniform(size=(5000, 2))
y_diag_prueba = (X_diag_prueba[:, 1] > X_diag_prueba[:, 0]).astype(int)

u1, u2 = np.meshgrid(np.linspace(0, 1, 300), np.linspace(0, 1, 300))
malla_u = np.c_[u1.ravel(), u2.ravel()]

fig, ejes = plt.subplots(1, 4, figsize=(20, 5))
for eje, d in zip(ejes, [2, 4, 8, None]):
    a = DecisionTreeClassifier(max_depth=d, random_state=SEMILLA).fit(X_diag, y_diag)
    z = a.predict(malla_u).reshape(u1.shape)
    eje.contourf(u1, u2, z, levels=[-0.5, 0.5, 1.5], cmap=FONDO)
    eje.plot([0, 1], [0, 1], color="black", linestyle="--", linewidth=1)
    for clase in (0, 1):
        eje.scatter(*X_diag[y_diag == clase].T, color=COLORES[clase], s=6)
    eje.set_aspect("equal")
    eje.set_title(f"profundidad {d or 'sin límite'}: {a.get_n_leaves()} hojas, "
                  f"prueba {a.score(X_diag_prueba, y_diag_prueba):.3f}")
plt.tight_layout()
plt.show()

Para una frontera que la logística representa con **una** recta, el árbol necesita 19
hojas, y aun sin límite de profundidad se equivoca en el 5 % de los puntos de prueba:
los que caen en los triángulos entre la escalera y la diagonal. Con más datos la
escalera tendría más peldaños, más finos, pero nunca sería una recta.

Ahora rotemos los datos 45°. Con las nuevas coordenadas
$v_1 = (x_1 + x_2)/\sqrt2$ y $v_2 = (x_2 - x_1)/\sqrt2$ la diagonal se vuelve el eje
$v_2 = 0$, y la frontera, un único corte horizontal:

In [ ]:
def rotar(X):
    return np.c_[X[:, 0] + X[:, 1], X[:, 1] - X[:, 0]] / np.sqrt(2)


a_rotado = DecisionTreeClassifier(random_state=SEMILLA).fit(rotar(X_diag), y_diag)
print(f"Árbol sobre los datos rotados: {a_rotado.get_n_leaves()} hojas, profundidad "
      f"{a_rotado.get_depth()}, accuracy en prueba {a_rotado.score(rotar(X_diag_prueba), y_diag_prueba):.4f}")
print(f"Corte: v{a_rotado.tree_.feature[0] + 1} <= {a_rotado.tree_.threshold[0]:.4f}")

Mismos datos, mismo problema, mismo algoritmo: dos hojas en lugar de 19, y un error que
pasa del 5 % a algo más de una milésima (el corte se estima con 400 puntos, así que no
cae exactamente en cero). La
lección práctica es que **un árbol no puede crear la variable $x_2 - x_1$**; si esa
diferencia es lo que importa, hay que dársela. Es ingeniería de características
(módulo 2) a favor del árbol, y también la razón por la que los ensambles de árboles a
veces pierden contra un modelo lineal cuando la señal es una suma de muchas variables.

## 7. La miopía del algoritmo voraz

El árbol elige cada corte mirando **solo un paso adelante**: el que más reduce la
impureza ahora, sin considerar qué cortes permitirá después. Casi siempre funciona. El
caso clásico en el que falla es el **XOR**: cuatro nubes en las esquinas de un cuadrado,
donde la clase depende de la **combinación** de las dos variables — aprueba si ambas son
altas o ambas son bajas.

In [ ]:
def generar_xor(n, rng):
    esquinas = rng.choice([-1, 1], size=(n, 2))
    X = esquinas + rng.normal(scale=0.45, size=(n, 2))
    y = (esquinas[:, 0] * esquinas[:, 1] > 0).astype(int)
    return X, y


X_xor, y_xor = generar_xor(400, rng)
X_xor_prueba, y_xor_prueba = generar_xor(5000, rng)

fig, ejes = plt.subplots(1, 3, figsize=(17, 4.8))
for clase in (0, 1):
    ejes[0].scatter(*X_xor[y_xor == clase].T, color=COLORES[clase], s=8)
ejes[0].set_aspect("equal")
ejes[0].set_title("XOR: la clase depende de la combinación")
for eje, j in zip(ejes[1:], [0, 1]):
    cands = candidatos(X_xor[:, j])
    reduccion = np.array([gini(y_xor) - gini_corte(X_xor[:, j], y_xor, t) for t in cands])
    eje.plot(cands, reduccion, color="black", linewidth=1)
    eje.plot(cands[reduccion.argmax()], reduccion.max(), "o", color="gold", markersize=9,
             markeredgecolor="black", label=f"máximo: {reduccion.max():.4f}")
    eje.axhline(gini(y_xor), color="gray", linestyle="--", linewidth=1,
                label=f"Gini sin cortar: {gini(y_xor):.3f}")
    eje.set_ylim(-0.01, 0.52)
    eje.legend(loc="center right", fontsize=9)
    eje.set_xlabel(f"umbral sobre x{j + 1}")
    eje.set_ylabel("reducción de Gini")
    eje.set_title(f"Primer corte sobre x{j + 1}: ninguno ayuda")
plt.tight_layout()
plt.show()

Ningún corte de una sola variable reduce la impureza de manera apreciable: el mejor de
todos baja el Gini menos de una centésima, partiendo de 0.5. A cada lado de cualquier
recta vertical u horizontal sigue habiendo mitad y mitad. Para el algoritmo
voraz, ninguna variable "sirve". Y sin embargo dos cortes —$x_1 \leq 0$ y luego
$x_2 \leq 0$ en cada lado— resuelven el problema casi perfectamente. Comparemos el árbol
voraz de profundidad 2 con ese árbol de profundidad 2 construido a mano:

In [ ]:
voraz2 = DecisionTreeClassifier(max_depth=2, random_state=SEMILLA).fit(X_xor, y_xor)


def arbol_ideal(X):
    return ((X[:, 0] > 0) == (X[:, 1] > 0)).astype(int)


filas = [
    {"árbol": "voraz, profundidad 2", "raíz": f"x{voraz2.tree_.feature[0] + 1} ≤ {voraz2.tree_.threshold[0]:.2f}",
     "accuracy en prueba": voraz2.score(X_xor_prueba, y_xor_prueba)},
    {"árbol": "a mano, profundidad 2", "raíz": "x1 ≤ 0",
     "accuracy en prueba": np.mean(arbol_ideal(X_xor_prueba) == y_xor_prueba)},
]
for d in [3, 4, None]:
    a = DecisionTreeClassifier(max_depth=d, random_state=SEMILLA).fit(X_xor, y_xor)
    filas.append({"árbol": f"voraz, profundidad {d or 'sin límite'} ({a.get_n_leaves()} hojas)",
                  "raíz": f"x{a.tree_.feature[0] + 1} ≤ {a.tree_.threshold[0]:.2f}",
                  "accuracy en prueba": a.score(X_xor_prueba, y_xor_prueba)})
detenido = DecisionTreeClassifier(min_impurity_decrease=0.01, random_state=SEMILLA).fit(X_xor, y_xor)
filas.append({"árbol": f"voraz con min_impurity_decrease=0.01 ({detenido.get_n_leaves()} hoja)",
              "raíz": "—", "accuracy en prueba": detenido.score(X_xor_prueba, y_xor_prueba)})
pd.DataFrame(filas).set_index("árbol").round(3)

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(11, 5))
x1, x2 = np.meshgrid(np.linspace(-2.6, 2.6, 300), np.linspace(-2.6, 2.6, 300))
malla_xor = np.c_[x1.ravel(), x2.ravel()]
for eje, (pred, titulo) in zip(ejes, [(voraz2.predict(malla_xor), "Voraz, profundidad 2"),
                                     (arbol_ideal(malla_xor), "A mano, profundidad 2")]):
    eje.contourf(x1, x2, pred.reshape(x1.shape), levels=[-0.5, 0.5, 1.5], cmap=FONDO)
    for clase in (0, 1):
        eje.scatter(*X_xor[y_xor == clase].T, color=COLORES[clase], s=6)
    eje.set_aspect("equal")
    eje.set_title(titulo)
plt.tight_layout()
plt.show()

El árbol voraz eligió como raíz el corte con la mayor reducción, que es casi cero y está
en un lugar arbitrario: $x_2 \leq 1.68$, que solo aparta a unos pocos puntos del borde
superior. Desde ahí, ni con profundidad 2, 3 o 4 pasa del 0.52 — una moneda. Sin límite
de profundidad termina encontrando la estructura (0.935), pero con 23 hojas y aun por
debajo de las 4 hojas del árbol a mano (0.971). Y una regla de parada razonable como
`min_impurity_decrease` —"no cortes si no mejora al menos 0.01"— lo detiene en la raíz:
un árbol de una hoja, que acierta la mitad.

Encontrar el árbol óptimo de verdad es un problema NP-completo (`04-arboles-y-bagging.md`);
el voraz es el compromiso que lo hace viable. Su punto ciego son las **interacciones
puras**: variables que no dicen nada solas y lo dicen todo juntas. Con datos reales es
raro que una interacción no deje ningún rastro en las variables individuales, y por eso
los árboles funcionan bien en la práctica, pero conviene saber dónde no miran.

## 8. 🔵 Opcional: árboles de regresión y los cuadrados del módulo 3

Un árbol también predice números: en cada hoja, la **media** de $y$ de los puntos que
caen en ella. Para elegir el corte, la impureza es la suma de cuadrados de los residuales
alrededor de esa media — el **mismo SSE** que `00-regresion-lineal-intuicion.ipynb`
(módulo 3) dibujó como un área de cuadrados. Usemos sus mismos 10 estudiantes:
`promedio_anterior` → `nota_final`.

In [ ]:
diez = datos.sample(n=10, random_state=SEMILLA).sort_values("promedio_anterior")
x10 = diez["promedio_anterior"].to_numpy()
y10 = diez["nota_final"].to_numpy()


def sse_corte(x, y, t):
    izq, der = y[x <= t], y[x > t]
    return np.sum((izq - izq.mean()) ** 2) + np.sum((der - der.mean()) ** 2)


cands10 = candidatos(x10)
sse_cands = np.array([sse_corte(x10, y10, t) for t in cands10])
t10 = cands10[np.argmin(sse_cands)]

tocon_reg = DecisionTreeRegressor(max_depth=1).fit(x10[:, None], y10)
recta = LinearRegression().fit(x10[:, None], y10)
sse_recta = np.sum((y10 - recta.predict(x10[:, None])) ** 2)
print(f"Mejor corte a mano: promedio <= {t10:.3f}   scikit-learn: {tocon_reg.tree_.threshold[0]:.3f}")
print(f"SSE — sin cortar (media): {np.sum((y10 - y10.mean()) ** 2):.3f}   "
      f"tocón: {sse_cands.min():.3f}   recta de mínimos cuadrados: {sse_recta:.3f}")

# Los mismos dos modelos, ajustados con los 10, evaluados sobre los otros 390 estudiantes
resto = datos.drop(diez.index)
x_resto, y_resto = resto[["promedio_anterior"]].to_numpy(), resto["nota_final"].to_numpy()
for nombre, modelo in [("tocón", tocon_reg), ("recta", recta)]:
    ecm = np.mean((y_resto - modelo.predict(x_resto)) ** 2)
    print(f"ECM en los otros 390 — {nombre}: {ecm:.3f}")

In [ ]:
LIM10_X, LIM10_Y = (2.4, 5.0), (1.9, 5.0)


def panel_cuadrados(eje, prediccion, titulo, dibujar_modelo):
    y_hat = prediccion(x10)
    for xi, yi, yh in zip(x10, y10, y_hat):
        lado = abs(yi - yh)
        eje.add_patch(Rectangle((xi, min(yi, yh)), lado, lado, facecolor="tab:red",
                                edgecolor="tab:red", alpha=0.25, zorder=1))
        eje.plot([xi, xi], [yi, yh], color="tab:red", linewidth=1.5, zorder=2)
    dibujar_modelo(eje)
    eje.scatter(x10, y10, color="black", zorder=3)
    eje.set_xlim(*LIM10_X)
    eje.set_ylim(*LIM10_Y)
    eje.set_aspect("equal")
    eje.set_xlabel("promedio_anterior")
    eje.set_ylabel("nota_final")
    eje.set_title(f"{titulo}\nSSE = {np.sum((y10 - y_hat) ** 2):.3f}")


xs = np.linspace(*LIM10_X, 500)
fig, ejes = plt.subplots(1, 3, figsize=(17, 5.8))
panel_cuadrados(ejes[0], lambda x: tocon_reg.predict(x[:, None]),
                f"Tocón: corte en {t10:.2f}",
                lambda eje: eje.step(xs, tocon_reg.predict(xs[:, None]), where="post",
                                     color="tab:purple", linewidth=2))
panel_cuadrados(ejes[1], lambda x: recta.predict(x[:, None]), "Recta de mínimos cuadrados",
                lambda eje: eje.plot(xs, recta.predict(xs[:, None]), color="tab:purple", linewidth=2))

ejes[2].plot(cands10, sse_cands, "o-", color="black")
ejes[2].axhline(sse_recta, color="tab:purple", linestyle="--", label="SSE de la recta")
ejes[2].axhline(np.sum((y10 - y10.mean()) ** 2), color="tab:green", linestyle="--",
                label="SSE de la media (sin cortar)")
ejes[2].set_xlabel("umbral t sobre promedio_anterior")
ejes[2].set_ylabel("SSE del tocón")
ejes[2].set_title("El SSE de cada corte posible")
ejes[2].legend()
plt.tight_layout()
plt.show()

El tocón reemplaza la recta por un **escalón**: dos medias, una a cada lado del corte, y
elige el corte que deja el área roja más pequeña. Con estos 10 estudiantes deja **menos**
área que la recta (0.93 frente a 1.49): elegir dónde cortar y dos medias es más flexible
que elegir pendiente e intercepto, y en entrenamiento la flexibilidad siempre gana. Fuera
de la muestra la ventaja desaparece: sobre los otros 390 estudiantes los dos quedan
prácticamente empatados (error cuadrático medio de 0.284 el tocón y 0.279 la recta). Un
37 % menos de área en entrenamiento no compró nada: era, en buena parte, ajuste a estos
10 puntos. Es la lección de sesgo-varianza del módulo 3 otra vez, en miniatura. Agregando cortes, el escalón se vuelve una escalera que puede seguir cualquier
forma, hasta que con una hoja por estudiante el SSE de entrenamiento llega a cero:

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(17, 5.8))
for eje, d in zip(ejes, [2, 3, None]):
    a = DecisionTreeRegressor(max_depth=d).fit(x10[:, None], y10)
    panel_cuadrados(eje, lambda x, a=a: a.predict(x[:, None]),
                    f"profundidad {d or 'sin límite'} ({a.get_n_leaves()} hojas)",
                    lambda eje, a=a: eje.step(xs, a.predict(xs[:, None]), where="post",
                                              color="tab:purple", linewidth=2))
plt.tight_layout()
plt.show()

Sin límite, cada estudiante tiene su propia hoja, el área roja desaparece y el $R^2$ de
entrenamiento es 1: la versión de regresión del árbol que memoriza. Los árboles de
regresión son además la pieza con la que se construye el gradient boosting del
notebook 05, incluso para clasificar: cada ronda ajusta un árbol de regresión a los
residuales.

## Resumen

| Lo que se vio | Su forma geométrica | Conecta con |
|---|---|---|
| Una pregunta $x_j \leq t$ | Un corte vertical u horizontal | `04-arboles-y-bagging.md`, sección 1 |
| Elegir $t$ | Mínimo de una curva **escalonada**; basta probar puntos medios | Búsqueda exhaustiva del notebook 03 |
| El árbol | Rectángulos dentro de rectángulos; cada hoja, una regla | Interpretabilidad (`06-interpretabilidad.md`) |
| La predicción | Terrazas planas, no una rampa | Bagging y boosting suavizan las terrazas (notebooks 03 y 05) |
| Invariancia al orden | Estandarizar o transformar no cambia los cortes | KNN sí cambia (notebook 01b); sin `StandardScaler` en el notebook 04 |
| Cortes paralelos a los ejes | Una diagonal es una escalera; rotada, un corte | Ingeniería de características (módulo 2) |
| Miopía voraz | En el XOR ningún primer corte ayuda | Reglas de parada; profundidad (notebook 03) |
| 🔵 Árbol de regresión | Escalones que minimizan el área de los cuadrados | `00-regresion-lineal-intuicion.ipynb` (módulo 3); gradient boosting (notebook 05) |